# 🎭 Meme & Sarcasm Understanding – Exploratory Data Analysis

**Project:** Meme and Sarcasm Understanding – Comparative Analysis of 3 Models  
**Dataset:** MMSD 2.0 (Multimodal Sarcasm Detection Dataset v2) + Sample Dataset  

This notebook performs:
1. Dataset overview and class distribution
2. Text length analysis
3. Vocabulary statistics
4. Sample image visualization
5. Label correlation analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

from preprocessing import generate_sample_dataset, clean_text, compute_text_stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline
print('Libraries loaded ✓')

## 1. Generate & Load Sample Dataset

In [ ]:
# Generate sample dataset if not present
SAMPLE_DIR = Path('../data/sample_dataset')
META_PATH  = SAMPLE_DIR / 'metadata.json'

if not META_PATH.exists():
    print('Generating sample dataset …')
    generate_sample_dataset(n_samples=200, seed=42)

with open(META_PATH) as f:
    records = json.load(f)

df = pd.DataFrame(records)
df['label_name'] = df['label'].map({0: 'Not Sarcastic', 1: 'Sarcastic'})
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#4CAF50','#F44336'], edgecolor='white', linewidth=1.5)
for i, (name, val) in enumerate(counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', fontsize=13, fontweight='bold')
axes[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_ylim(0, max(counts.values) + 15)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4CAF50','#F44336'], startangle=140,
            textprops={'fontsize': 12})
axes[1].set_title('Class Distribution (Proportion)', fontsize=13, fontweight='bold')

plt.suptitle('MMSD Sample Dataset – Class Balance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/graphs/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(counts)

## 3. Text Length Analysis

In [ ]:
df['text_len']       = df['text'].apply(lambda x: len(x.split()))
df['clean_text_len'] = df['clean_text'].apply(lambda x: len(x.split()))
df['char_len']       = df['text'].apply(len)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
    ['text_len', 'clean_text_len', 'char_len'],
    ['Word Count (Raw)', 'Word Count (Cleaned)', 'Character Count']):
    for label, grp in df.groupby('label_name'):
        ax.hist(grp[col], bins=15, alpha=0.6, label=label,
                color='#4CAF50' if label == 'Not Sarcastic' else '#F44336')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Length')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Text Length Distribution by Class', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/graphs/eda_text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print(df[['text_len', 'clean_text_len', 'char_len']].describe().round(2))

## 4. Vocabulary Analysis

In [ ]:
sarc_texts  = ' '.join(df[df['label']==1]['clean_text'])
norm_texts  = ' '.join(df[df['label']==0]['clean_text'])
all_texts   = ' '.join(df['clean_text'])

sarc_words  = Counter(sarc_texts.split())
norm_words  = Counter(norm_texts.split())
all_words   = Counter(all_texts.split())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, counter, title, color in zip(
    axes,
    [sarc_words, norm_words],
    ['Top 15 Words – Sarcastic', 'Top 15 Words – Not Sarcastic'],
    ['#F44336', '#4CAF50']
):
    top = counter.most_common(15)
    words, freqs = zip(*top)
    ax.barh(list(words)[::-1], list(freqs)[::-1], color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Most Common Words by Class', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/graphs/eda_word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

stats = compute_text_stats(df['clean_text'].tolist())
print('\nVocabulary Statistics:')
for k, v in stats.items():
    print(f'  {k:<15}: {v}')

## 5. Sample Image Visualization

In [ ]:
IMG_DIR = SAMPLE_DIR / 'images'

sarc_samples = df[df['label']==1].sample(4, random_state=42)
norm_samples = df[df['label']==0].sample(4, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row_idx, (samples, row_label) in enumerate(
        [(sarc_samples, 'Sarcastic'), (norm_samples, 'Not Sarcastic')]):
    for col_idx, (_, rec) in enumerate(samples.iterrows()):
        ax = axes[row_idx][col_idx]
        img_path = IMG_DIR / rec['image_file']
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
        ax.set_title(rec['text'][:30] + '…' if len(rec['text']) > 30 else rec['text'],
                     fontsize=8, wrap=True)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=11, fontweight='bold', rotation=90,
                          labelpad=10, color='#F44336' if row_label=='Sarcastic' else '#4CAF50')

fig.suptitle('Sample Meme Images from Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Dataset Summary Statistics

In [ ]:
print('=' * 50)
print('  DATASET SUMMARY')
print('=' * 50)
print(f"Total samples    : {len(df)}")
print(f"Sarcastic        : {(df['label']==1).sum()} ({(df['label']==1).mean()*100:.1f}%)")
print(f"Not Sarcastic    : {(df['label']==0).sum()} ({(df['label']==0).mean()*100:.1f}%)")
print(f"Avg text length  : {df['text_len'].mean():.1f} words")
print(f"Max text length  : {df['text_len'].max()} words")
print(f"Vocabulary size  : {len(set(all_texts.split()))}")
print('=' * 50)

# Correlation heatmap
numeric_cols = df[['label', 'text_len', 'clean_text_len', 'char_len']]
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.3f',
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()